# Street data with topology errors

This notebook creates `digiroad_topology_errors.gpkg`, which is used in Chapter 8.3 to demonstrate how to check and fix the topology of street data. The data is a part of `digiroad_helsinki.gpkg` (the Digiroad street data of Chapter 8, see `prepare_digiroad_network_data.ipynb`) in which we introduce three types of errors:

- **Missing nodes**: at three T-junctions, the two segments of the through street are joined into one line, so the side street ends on the middle of a line instead of at a node.
- **Gaps**: at three other T-junctions, the through street is joined in the same way and the side street is also shortened by 3 meters, so it ends before reaching the through street.
- **Duplicates**: three segments are copied, so the same street is in the data twice.

In [1]:
from pathlib import Path

import geopandas as gpd
import pandas as pd
import shapely
from shapely.ops import substring

data_dir = Path("../../part2/chapter-08/nb/data")
streets = gpd.read_file(data_dir / "digiroad_helsinki.gpkg")

OMP: Warning #179: Function Can't set size of /tmp file failed:


Select the segments that intersect a box covering Kamppi, Punavuori, Ruoholahti and Jätkäsaari (segments crossing the edge of the box are kept whole), keep only the `id` and `name` columns, and drop the Z coordinate:

In [2]:
area = gpd.GeoSeries(
    [shapely.box(24.905, 60.157, 24.945, 60.170)], crs="EPSG:4326"
).to_crs(streets.crs)
streets = streets.loc[streets.intersects(area.iloc[0]), ["id", "name", "geometry"]]
streets["geometry"] = shapely.force_2d(streets.geometry)
streets = streets.set_index("id", drop=False)
len(streets)

736

The T-junctions are given as the `id` values of the two through-street segments and of the side street:

In [3]:
missing_nodes = [
    ((625, 1813), 1075),  # Messitytönkuja / Laivapojankuja
    ((1821, 104), 1248),  # Kellosaarenkatu / Loistokuja
    ((2043, 1173), 1481),  # Saukonlaituri / Saukonkatu
]
gaps = [
    ((1869, 2041), 254),  # Lastenkodinkatu / Lastenkodin kuja
    ((1870, 1079), 1346),  # Laivurinkatu / Sepänkatu
    ((670, 1342), 829),  # Selkämerenkatu / Messitytönkatu
]
duplicate_ids = [1733, 1317, 1687]  # Annankatu, Laivurinkatu, Salmisaarenkatu

In [4]:
def join_through_street(streets, ids):
    """Joins the two segments ids into one line, which removes the node between them."""
    joined = shapely.line_merge(shapely.union_all(streets.loc[list(ids), "geometry"]))
    assert joined.geom_type == "LineString"
    streets.loc[ids[0], "geometry"] = joined
    return streets.drop(index=ids[1])


def shorten_end(line, other, distance):
    """Cuts distance meters off the end of line that touches the line other."""
    start, end = shapely.get_point(line, 0), shapely.get_point(line, -1)
    if start.distance(other) < end.distance(other):
        return substring(line, distance, line.length)
    return substring(line, 0, line.length - distance)


for through_ids, side_id in missing_nodes + gaps:
    streets = join_through_street(streets, through_ids)

for through_ids, side_id in gaps:
    through = streets.loc[through_ids[0], "geometry"]
    side = streets.loc[side_id, "geometry"]
    streets.loc[side_id, "geometry"] = shorten_end(side, through, 3)

streets = pd.concat([streets, streets.loc[duplicate_ids]])
len(streets)

733

In [5]:
streets = streets.reset_index(drop=True)
streets.to_file(data_dir / "digiroad_topology_errors.gpkg")